# ANALISE EXPLORATÓRIA

In [9]:
import pandas as pd 
import numpy as np 
import openpyxl
import subprocess

caminho_arquivo = "DRE 2019.xlsx"

df_plano = pd.read_excel(caminho_arquivo, sheet_name="Plano de Contas") #abrir exclusivamente a aba correspondente da planilha
df_formato = pd.read_excel(caminho_arquivo, sheet_name="Formato DRE")
df_realizado = pd.read_excel(caminho_arquivo, sheet_name="Realizado")
df_orcado = pd.read_excel(caminho_arquivo, sheet_name="Orçado")

display(df_plano.head())
display(df_formato.head())
display(df_realizado.head())
display(df_orcado.head())

,Conta,Descrição da Conta,Nível 1,Nível 2
0,999015461,Venda de Smart TV 50' 4K,(+) RECEITA BRUTA,01 - Vendas de Produtos
1,999149616,Venda de iPhone 7,(+) RECEITA BRUTA,01 - Vendas de Produtos
2,999113857,Venda de Galaxy S10,(+) RECEITA BRUTA,01 - Vendas de Produtos
3,999085686,Venda de Apple Watch,(+) RECEITA BRUTA,01 - Vendas de Produtos
4,999048672,Venda de Câmera Digital Rebel T6,(+) RECEITA BRUTA,01 - Vendas de Produtos


,Ordem,Grupo,Subtotal?
0,1,(+) RECEITA BRUTA,0
1,2,(-) Deduções da Receita,0
2,3,(=) RECEITA LÍQUIDA,1
3,4,(-) Custos de Vendas,0
4,5,(=) LUCRO BRUTO,1


,Mês/Ano,Conta,Valor Realizado
0,2019-01-01,999015461,810137
1,2019-01-01,999149616,850100
2,2019-01-01,999113857,299588
3,2019-01-01,999085686,506754
4,2019-01-01,999048672,586151


,Mês/Ano,Conta,Valor Orçado
0,2019-01-01,999015461,924509
1,2019-01-01,999149616,808117
2,2019-01-01,999113857,380859
3,2019-01-01,999085686,691242
4,2019-01-01,999048672,585387


# Descobrir a receita realizada

In [10]:
df_analise = df_realizado.merge(
    df_plano,
    on="Conta",
    how="left"
)

display(df_analise.head())

,Mês/Ano,Conta,Valor Realizado,Descrição da Conta,Nível 1,Nível 2
0,2019-01-01,999015461,810137,Venda de Smart TV 50' 4K,(+) RECEITA BRUTA,01 - Vendas de Produtos
1,2019-01-01,999149616,850100,Venda de iPhone 7,(+) RECEITA BRUTA,01 - Vendas de Produtos
2,2019-01-01,999113857,299588,Venda de Galaxy S10,(+) RECEITA BRUTA,01 - Vendas de Produtos
3,2019-01-01,999085686,506754,Venda de Apple Watch,(+) RECEITA BRUTA,01 - Vendas de Produtos
4,2019-01-01,999048672,586151,Venda de Câmera Digital Rebel T6,(+) RECEITA BRUTA,01 - Vendas de Produtos


# ENCONTRAR A RECEITA BRUTA

In [11]:

# Filtra somente as contas classificadas como Receita Bruta

df_receita = df_analise[
    df_analise["Nível 1"] == "(+) RECEITA BRUTA"
]

display(df_receita.head())

,Mês/Ano,Conta,Valor Realizado,Descrição da Conta,Nível 1,Nível 2
0,2019-01-01,999015461,810137,Venda de Smart TV 50' 4K,(+) RECEITA BRUTA,01 - Vendas de Produtos
1,2019-01-01,999149616,850100,Venda de iPhone 7,(+) RECEITA BRUTA,01 - Vendas de Produtos
2,2019-01-01,999113857,299588,Venda de Galaxy S10,(+) RECEITA BRUTA,01 - Vendas de Produtos
3,2019-01-01,999085686,506754,Venda de Apple Watch,(+) RECEITA BRUTA,01 - Vendas de Produtos
4,2019-01-01,999048672,586151,Venda de Câmera Digital Rebel T6,(+) RECEITA BRUTA,01 - Vendas de Produtos


# RECEITA POR MÊS

In [12]:
receita_mensal = (
    df_receita
    .groupby("Mês/Ano")["Valor Realizado"]
    .sum()
)

display(receita_mensal)

Mês/Ano
2019-01-01    20050916
2019-02-01    19648815
2019-03-01    21095352
2019-04-01    19936804
2019-05-01    17462474
2019-06-01    19786714
2019-07-01    20728326
2019-08-01    21262064
2019-09-01    21001843
2019-10-01    19216027
2019-11-01    19431711
2019-12-01    21131071
Name: Valor Realizado, dtype: int64

# RANKING DE PRODUTOS

In [13]:
# Agrupa os valores pela descrição do produto


receita_produto = (
    df_receita
    .groupby("Descrição da Conta")["Valor Realizado"]
    .sum()
)

# Ordena do maior para o menor.

receita_produto = receita_produto.sort_values(
    ascending=False
)

display(receita_produto)

Descrição da Conta
Venda de Moto G7                      8713680
Venda de Câmera Digital Rebel T6      8563185
Venda de Smart TV 75' 4K              8340122
Venda de Apple Watch                  8291974
Venda de Smart TV LED Full HD 55'     7931061
Venda de Inspiron 15                  7829374
Venda de Smart TV 50' 4K              7791574
Venda de Câmera Digital Sony Cyber    7774405
Venda de iPhone 7                     7768159
Venda de iPhone 6S                    7741927
Venda de Smart Watch MI               7559250
Venda de Smart Watch Galaxy           7466015
Venda de Dell G7                      7409118
Venda de Galaxy S10                   7368112
Venda de Câmera Coolpix L340          7262511
Venda de Galaxy J8                    7158168
Venda de iPad 32GB Wifi               7152703
Venda de Tablet M10 Android           7094319
Venda de Moto Z                       6918743
Venda de Galaxy S8                    6837973
Venda de iPhone XS                    6559192
Venda de TV LED

# ORÇADO X REALIZADO

In [14]:
# Junta as tabelas realizado e orçado

df_comparacao = df_realizado.merge(
    df_orcado,
    on=["Mês/Ano", "Conta"],
    how="left"
)

display(df_comparacao.head())

,Mês/Ano,Conta,Valor Realizado,Valor Orçado
0,2019-01-01,999015461,810137,924509
1,2019-01-01,999149616,850100,808117
2,2019-01-01,999113857,299588,380859
3,2019-01-01,999085686,506754,691242
4,2019-01-01,999048672,586151,585387


# CRIAR DESVIO

In [15]:
# Criar uma nova coluna chamada "Desvio".
# Desvio = Realizado - Orçado

df_comparacao["Desvio"] = (
    df_comparacao["Valor Realizado"]
    - df_comparacao["Valor Orçado"]
)

df_comparacao["Desvio %"] = (
    df_comparacao["Desvio"]
    / df_comparacao["Valor Orçado"]
) * 100


display (df_comparacao.head())

,Mês/Ano,Conta,Valor Realizado,Valor Orçado,Desvio,Desvio %
0,2019-01-01,999015461,810137,924509,-114372,-12.371107
1,2019-01-01,999149616,850100,808117,41983,5.195164
2,2019-01-01,999113857,299588,380859,-81271,-21.338868
3,2019-01-01,999085686,506754,691242,-184488,-26.689350
4,2019-01-01,999048672,586151,585387,764,0.130512


# CONVERTER TABELA EM ARQUIVO CSV

In [18]:
# Nome do arquivo CSV que será criado
arquivo_csv = "Analise_DRE.csv"

# Exporta o DataFrame df_comparacao para CSV
df_comparacao.to_csv(
    arquivo_csv,
    index=False,          # Não exporta o índice 0, 1, 2, 3...
    encoding="utf-8-sig", # Mantém corretamente os acentos
    sep=";"               # Separador utilizado no padrão brasileiro
)

print(f"CSV criado com sucesso: {arquivo_csv}")

CSV criado com sucesso: Analise_DRE.csv
